# 03 – Model Training & Evaluation

**Project:** Area Feasibility Scoring Model  
**Goal:** Train and compare a logistic regression baseline and a Random Forest model to predict area affordability.

---
### Contents
1. Setup
2. Data Preparation & Train/Test Split
3. Pipeline Construction (leakage-safe)
4. Cross-Validation – Model Selection
5. Final Evaluation on Held-Out Test Set
6. Feature Importances
7. ROC Curves
8. Predicting for a New User Budget
9. Summary

## 1 · Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

from data_loader import prepare_dataset
from features import (
    FEATURE_COLS, CLASSIFICATION_TARGET,
    assert_no_leakage, build_target,
)
from train import (
    make_logistic_pipeline, make_rf_pipeline,
    cross_validate_pipeline, print_cv_results,
    RANDOM_STATE, TEST_SIZE, CV_FOLDS,
)
from evaluate import (
    evaluate_classifier, print_classifier_metrics,
    compare_models, get_feature_importances,
    plot_feature_importances, plot_roc_curves,
)

USER_BUDGET = 300_000

## 2 · Data Preparation & Train/Test Split

In [ ]:
# Load / generate area-level data
area_df = prepare_dataset(user_budget=USER_BUDGET, use_synthetic=True)

# Build target BEFORE splitting (no statistics involved – just a comparison)
y = build_target(area_df)

# Leakage guard
assert_no_leakage(FEATURE_COLS, [CLASSIFICATION_TARGET, 'median_price'])

# Stratified split: 80% train, 20% test
# NOTE: we pass the raw area_df (not the engineered features) so that
# the Pipeline's featurizer can be fit exclusively on X_train.
X_train, X_test, y_train, y_test = train_test_split(
    area_df, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Train affordability rate: {y_train.mean():.1%}')
print(f'Test  affordability rate: {y_test.mean():.1%}')

## 3 · Pipeline Construction

Each pipeline embeds the `LeakageSafeFeaturizer` as the **first step**.
This guarantees that when sklearn runs cross-validation, the featurizer
is always re-fit on the CV training fold and never sees the validation fold.

In [ ]:
lr_pipeline = make_logistic_pipeline(C=1.0)
rf_pipeline = make_rf_pipeline(n_estimators=200)

print('Logistic Regression pipeline steps:')
for name, step in lr_pipeline.steps:
    print(f'  {name}: {type(step).__name__}')

print('\nRandom Forest pipeline steps:')
for name, step in rf_pipeline.steps:
    print(f'  {name}: {type(step).__name__}')

## 4 · Cross-Validation – Model Selection

Both pipelines are cross-validated on the **training set only**.
The test set remains unseen until final evaluation.

In [ ]:
print(f'Running {CV_FOLDS}-fold stratified cross-validation …\n')

lr_cv = cross_validate_pipeline(lr_pipeline, X_train, y_train, cv=CV_FOLDS)
print_cv_results('Logistic Regression (baseline)', lr_cv)

rf_cv = cross_validate_pipeline(rf_pipeline, X_train, y_train, cv=CV_FOLDS)
print_cv_results('Random Forest', rf_cv)

In [ ]:
# Side-by-side CV comparison bar chart
cv_data = {
    'Logistic Regression': {k: v[0] for k, v in lr_cv.items()},
    'Random Forest':       {k: v[0] for k, v in rf_cv.items()},
}
cv_df = pd.DataFrame(cv_data).T

cv_df.plot(kind='bar', figsize=(10, 5), ylim=(0, 1.05),
           color=['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6'],
           edgecolor='white')
plt.title('Cross-Validation Scores: Baseline vs Random Forest')
plt.xlabel('Model')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5 · Final Evaluation on Held-Out Test Set

Refit on the full training set, then evaluate once on the held-out test set.

In [ ]:
print('Fitting final models on full training set …')
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
print('Done.')

In [ ]:
lr_metrics = evaluate_classifier(lr_pipeline, X_test, y_test, 'Logistic Regression')
rf_metrics = evaluate_classifier(rf_pipeline, X_test, y_test, 'Random Forest')

print_classifier_metrics(lr_metrics)
print_classifier_metrics(rf_metrics)

In [ ]:
# Model comparison table
comparison = compare_models([lr_metrics, rf_metrics])
print('\nModel Comparison (Test Set):')
comparison

## 6 · Feature Importances

In [ ]:
lr_imp = get_feature_importances(lr_pipeline)
rf_imp = get_feature_importances(rf_pipeline)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp, title in zip(
    axes,
    [lr_imp, rf_imp],
    ['Logistic Regression – |Coefficient|', 'Random Forest – Feature Importance'],
):
    ax.barh(imp['feature'][::-1], imp['importance'][::-1],
            color='steelblue', edgecolor='white')
    ax.set_xlabel('Importance')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 7 · ROC Curves

In [ ]:
plot_roc_curves(
    {'Logistic Regression': lr_pipeline, 'Random Forest': rf_pipeline},
    X_test,
    y_test,
)

## 8 · Predicting for a New User Budget

Demonstrate how to use the trained pipeline to score areas for a given user.

In [ ]:
# Suppose a new user has a different budget
NEW_BUDGET = 250_000

# Reuse the same area statistics but update the budget column
new_user_df = area_df.copy()
new_user_df['budget'] = NEW_BUDGET

# Predict using the fitted Random Forest pipeline
predictions  = rf_pipeline.predict(new_user_df)
probabilities = rf_pipeline.predict_proba(new_user_df)[:, 1]

results = new_user_df[['area', 'median_price', 'num_listings']].copy()
results['affordable']   = predictions
results['prob_affordable'] = probabilities.round(3)
results['budget'] = NEW_BUDGET

print(f'User budget: £{NEW_BUDGET:,.0f}')
print(f'Affordable areas: {predictions.sum()} / {len(predictions)}')
results.sort_values('prob_affordable', ascending=False).head(10)

## 9 · Summary

_Update after running:_

| | Logistic Regression | Random Forest |
|---|---|---|
| CV Accuracy | | |
| CV ROC-AUC | | |
| Test Accuracy | | |
| Test ROC-AUC | | |
| Top feature | | |

**Key observations:**
- Both models perform well because `affordability_ratio` / `budget_deficit` are highly predictive.
- Random Forest typically outperforms on non-linear boundaries.
- Feature engineering (especially `pct_within_budget` and `ratio_x_spread`) adds signal beyond raw ratios.
- No data leakage: the featurizer is always fit on training data only (enforced by Pipeline).